# Entwurf nichtrekursiver Filter mittels Fensterverfahren

Der komplette Filterentwurf durch Fensterung der Impulsantwort eines zeitverschobenen idealen Tiefpassfilters wird in Python mit Bibliotheken wie `scipy.signal.firwin` umgesetzt.

Ein Aufruf von `b = firwin(m+1, Wn)` erzeugt $m+1$ Filterkoeffizienten eines Tiefpassfilters der Ordnung `m` mit Grenzfrequenz `Wn`.

Für einen Hochpassfilter: `b = firwin(m+1, Wn, pass_zero=False)` erzeugt den entsprechenden Hochpassfilter mit Grenzfrequenz `Wn`.

Für einen Bandpassfilter: `b = firwin(m+1, [Wn1, Wn2], pass_zero=False)` erzeugt einen Bandpassfilter mit den Grenzfrequenzen `Wn1` und `Wn2`.

Standardmäßig wird das **Hamming-Fenster** verwendet. Zusätzlich kann der letzte Parameter ein Vektor mit Samples eines beliebigen Fensterfunktion sein: `b = firwin(m+1, Wn, window=window)`. Dieser Vektor muss $m+1$ Samples enthalten.

---

### a)

Entwerfen Sie einen **Tiefpassfilter**, indem Sie `m = 50` und `Wn = 0.4` setzen.
Vergleichen Sie den Absolutwert des Spektrums für verschiedene Fensterfunktionen.

---

### b)

Entwerfen Sie einen **Hochpassfilter** der Ordnung `m = 33` und Grenzfrequenz `Wn = 0.4` unter Verwendung eines Hamming-Fensters.
Warum erzeugt Python/Scipy hier eine Warnung?

---

### c)

Gruppenlaufzeit und Linearität eines Systems stehen in engem Zusammenhang.
Berechnen Sie die **Gruppenlaufzeit** des Tiefpassfilters aus Aufgabe a) (Hamming-Fenster).
Verwenden Sie dazu Python, z. B. `scipy.signal.group_delay`.

In [ ]:
# BEGIN SOLUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import firwin, freqz, group_delay
from scipy.signal.windows import kaiser

# ----------------------------------------------------------
# Task a: Lowpass Filter Design
# ----------------------------------------------------------

N = 50
cutoff = 0.4

# Windows (MATLAB-like)
box   = np.ones(N+1)
hann  = np.hanning(N+1)
hamm  = np.hamming(N+1)
black = np.blackman(N+1)
kais  = kaiser(N+1, beta=8)

# Base lowpass FIR (no window)
b_base = firwin(N+1, cutoff)

# Apply MATLAB-like windows manually
bR  = b_base * box
bH  = b_base * hann
bHM = b_base * hamm
bB  = b_base * black
bK  = b_base * kais

# Frequency responses
W, HR  = freqz(bR, 1, 512)
_, HH  = freqz(bH, 1, 512)
_, HHM = freqz(bHM, 1, 512)
_, HB  = freqz(bB, 1, 512)
_, HK  = freqz(bK, 1, 512)

# ----------------------------------------------------------
# Plot 1: Rectangular, Hann, Hamming
# ----------------------------------------------------------

plt.figure()
plt.plot(W/np.pi, 20*np.log10(np.abs(HR / np.max(HR))))
plt.plot(W/np.pi, 20*np.log10(np.abs(HH / np.max(HH))), '--')
plt.plot(W/np.pi, 20*np.log10(np.abs(HHM / np.max(HHM))), '-.')
plt.xlabel('Omega/pi')
plt.ylabel('|H(exp(j*Omega))/H(1)| in dB')
plt.grid()
plt.title('Lowpass: - Rectangular   -- Hann   -. Hamming')
plt.axis([0, 1, -150, 0])

# ----------------------------------------------------------
# Plot 2: Blackman, Kaiser
# ----------------------------------------------------------

plt.figure()
plt.plot(W/np.pi, 20*np.log10(np.abs(HB / np.max(HB))), '-')
plt.plot(W/np.pi, 20*np.log10(np.abs(HK / np.max(HK))), '--')
plt.xlabel('Omega/pi')
plt.ylabel('|H(exp(j*Omega))/H(1)| in dB')
plt.grid()
plt.title('Lowpass: - Blackman   -- Kaiser β=8')
plt.axis([0, 1, -150, 0])

# ----------------------------------------------------------
# Task b: Highpass Filter
# ----------------------------------------------------------

N_hp = 33
b_hp = firwin(N_hp, cutoff, pass_zero=False)
W, H = freqz(b_hp, 1, 512)

plt.figure()
plt.plot(W/np.pi, 20*np.log10(np.abs(H / np.max(H))))
plt.grid()
plt.xlabel('Omega/pi')
plt.ylabel('|H(exp(j*Omega))/H(-1)| in dB')
plt.title('Highpass')

# ----------------------------------------------------------
# Task c: Group Delay
# ----------------------------------------------------------

b_gd = firwin(50, cutoff)
w, gd = group_delay((b_gd, 1), 256)

plt.figure()
plt.plot(w/np.pi, gd)
plt.grid()
plt.title('Group Delay in Samples')
plt.xlabel('Omega/pi')
plt.axis([0, 1, 24, 26])

plt.show()
# END SOLUTION

## Lösung: Entwurf nichtrekursiver Filter mittels Fensterverfahren

### a)

In den folgenden Abbildungen sind die **Amplitudenverläufe** der einzelnen Filter mit Ordnung $m=50$ dargestellt.

<img src="./pngs/ldsv5a02_a1.png" width="250">
<img src="./pngs/ldsv5a02_a2.png" width="250">

Die Abbildungen zeigen die normalisierte Grenzfrequenz `Wn=0.4`, die für alle Filterentwürfe verwendet wurde.
Ein Vergleich der Amplitudenverläufe verdeutlicht das Problem, einen Kompromiss zwischen höchstmöglicher Dämpfung der ersten Nebenkeule und möglichst kleinem Durchlassband zu finden.
Betrachtet man den Trend der Kaiser- gegenüber der Blackman-Fensterung, ist der freie Parameter $\beta$ von Vorteil.
Bei nahezu gleicher Flankensteilheit hat das Kaiser-Fenster ein kleineres Durchlassband ($\Omega_S/\pi \approx 0.5$ vs. $\Omega_S/\pi \approx 0.55$) und gleichzeitig eine bessere erste Nebenkeule ($a_N \approx 83,\text{dB}$ vs. $a_N \approx 75,\text{dB}$).

---

### b)

Die dargestellte Kurve zeigt die Amplitudencharakteristik eines **Hochpassfilters** mit Ordnung $m=33$ und einer normalisierten Grenzfrequenz von $0.4$.
Verwendet wird hier ein Hamming-Fenster.

<img src="./pngs/ldsv5a02_b1.png" width="250">

Beim Entwurf eines Hochpasses mit der Python-Funktion `scipy.signal.firwin()` erzeugt Python folgende Warnung:

```
For highpass and bandstop filters, order must be even.
Order is being increased by 1.
```

Die Funktion `firwin()` entwirft Filter vom Typ I oder II, abhängig davon, ob die Ordnung gerade oder ungerade ist.
Bei ungerader Ordnung (Typ II) muss die Frequenzantwort bei der Nyquist-Frequenz $\Omega/\pi=1$ null sein, was dem geforderten Hochpassverhalten widerspricht.
Daher erhöht Python automatisch die Filterordnung (Typ I).

---

### c)

Wenn die **Gruppenlaufzeit** mit Python (z. B. `scipy.signal.group_delay()`) berechnet wird, ergibt sich folgender Verlauf über die normalisierte Frequenz für den entworfenen Tiefpass aus Aufgabe a) mit $m=50$ (Fenster: Hamming):

<img src="./pngs/ldsv5a02_c1.png" width="250">

Die Abbildung zeigt eine konstante Gruppenlaufzeit und verdeutlicht die geforderte Linearität des Systems.